# E7 - Al-Kafri axial: licencia y subset curado

Este notebook audita formalmente la licencia/viabilidad de Al-Kafri/Sudirman para el PFI publico y construye, solo si hay evidencia suficiente, un subset axial curado de alta confianza. No entrena modelos, no usa datasets restrictivos como fuente experimental y no fuerza pares ambiguos.

In [ ]:
try:
    import pydicom  # noqa: F401
    import skimage  # noqa: F401
except Exception:
    %pip install -q pydicom scikit-image

In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from scipy import ndimage
from skimage.transform import resize
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 180)

## Montaje de Drive y rutas

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No se monto Google Drive automaticamente. Si estas en Colab, montalo manualmente.")
    print("Detalle:", repr(exc))

In [ ]:
ALKAFRI_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI")
RAW_ZIPS = ALKAFRI_ROOT / "raw_zips"
EXTRACTED_ROOT = ALKAFRI_ROOT / "extracted"
NESTED_EXTRACTED_ROOT = EXTRACTED_ROOT / "_nested"
INVENTORY_RESULTS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory")
PAIRING_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing")
PAIRING_VALIDATION_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing_validation")
CURATION_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E7_alkafri_axial_curated_subset")
CURATED_PROCESSED_ROOT = ALKAFRI_ROOT / "processed" / "axial_curated_v1"
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [CURATION_ROOT, CURATED_PROCESSED_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

MAX_CURATION_CANDIDATES = 150
MIN_CURATED_PAIRS = 30
RANDOM_SEED = 42

print("CURATION_ROOT:", CURATION_ROOT)

## Auditoria de licencia

In [ ]:
license_rows = [
    {
        "dataset_name": "Lumbar Spine MRI Dataset",
        "source_url": "https://data.mendeley.com/datasets/k57fr854j2/2",
        "doi": "10.17632/k57fr854j2.2",
        "license": "CC BY 4.0",
        "access_type": "public_download",
        "redistributable": True,
        "academic_use": True,
        "notes": "MRI anonimizada de 515 pacientes; incluye cortes sagitales y axiales.",
        "decision_for_pfi": "usable",
    },
    {
        "dataset_name": "Label Image Ground Truth Data for Lumbar Spine MRI Dataset",
        "source_url": "https://data.mendeley.com/datasets/zbf6b4pttk/2",
        "doi": "10.17632/zbf6b4pttk.2",
        "license": "CC BY 4.0",
        "access_type": "public_download",
        "redistributable": True,
        "academic_use": True,
        "notes": "Label images/ground truth axial para SegNet: IVD, PE, TS y AAP.",
        "decision_for_pfi": "usable",
    },
    {
        "dataset_name": "Radiologists Notes for Lumbar Spine MRI Dataset",
        "source_url": "https://data.mendeley.com/datasets/s6bgczr8s2/2",
        "doi": "10.17632/s6bgczr8s2.2",
        "license": "CC BY 4.0",
        "access_type": "public_download",
        "redistributable": True,
        "academic_use": True,
        "notes": "Notas de radiologos; usar como metadata/referencia, no como labels de entrenamiento automatico.",
        "decision_for_pfi": "usable_as_metadata_reference",
    },
    {
        "dataset_name": "MATLAB source code for Lumbar Spine MRI Dataset",
        "source_url": "https://data.mendeley.com/datasets/8cp2cp7km8/2",
        "doi": "10.17632/8cp2cp7km8.2",
        "license": "GPLv3",
        "access_type": "public_download",
        "redistributable": True,
        "academic_use": True,
        "notes": "Usar solo como referencia de estructura/pairing; evitar copiar codigo MATLAB al producto final.",
        "decision_for_pfi": "reference_only_avoid_product_code_copy",
    },
    {
        "dataset_name": "RSNA/LumbarDISC",
        "source_url": "MIRA/Kaggle restricted sources",
        "doi": "",
        "license": "non-commercial/restrictive",
        "access_type": "controlled_or_competition",
        "redistributable": False,
        "academic_use": "reference_only",
        "notes": "No usar como dataset experimental principal del PFI publico.",
        "decision_for_pfi": "reference_only_not_experimental_dataset",
    },
    {
        "dataset_name": "SPIDER",
        "source_url": "project documented source",
        "doi": "",
        "license": "CC BY 4.0",
        "access_type": "public_download",
        "redistributable": True,
        "academic_use": True,
        "notes": "Dataset principal sagital ya validado en MVP.",
        "decision_for_pfi": "main_sagittal_dataset",
    },
    {
        "dataset_name": "SSMSpine/SymTC",
        "source_url": "https://github.com/jiasongchen/SSMSpine",
        "doi": "",
        "license": "research/synthetic repository",
        "access_type": "public_repository",
        "redistributable": "review_repository_terms",
        "academic_use": True,
        "notes": "Sintetico/mid-sagittal; no resuelve axial.",
        "decision_for_pfi": "supplemental_benchmark_or_pretraining_only",
    },
]
dataset_license_audit_df = pd.DataFrame(license_rows)
license_audit_csv_path = CURATION_ROOT / "E7_alkafri_dataset_license_audit.csv"
dataset_license_audit_df.to_csv(license_audit_csv_path, index=False)
display(dataset_license_audit_df)

## Cargar outputs previos

In [ ]:
INPUTS = {
    "extracted_inventory": INVENTORY_RESULTS_ROOT / "E6_alkafri_extracted_file_inventory.csv",
    "series_orientation": INVENTORY_RESULTS_ROOT / "E6_alkafri_series_orientation_candidates.csv",
    "ground_truth_inventory": INVENTORY_RESULTS_ROOT / "E6_alkafri_ground_truth_inventory.csv",
    "inventory_report": INVENTORY_RESULTS_ROOT / "E6_alkafri_inventory_report.json",
    "axial_ima_candidates": PAIRING_ROOT / "E6_alkafri_axial_ima_candidates.csv",
    "axial_series_summary": PAIRING_ROOT / "E6_alkafri_axial_series_summary.csv",
    "ground_truth_real_files": PAIRING_ROOT / "E6_alkafri_ground_truth_real_files.csv",
    "ground_truth_path_tokens": PAIRING_ROOT / "E6_alkafri_ground_truth_path_tokens.csv",
    "image_path_tokens": PAIRING_ROOT / "E6_alkafri_image_path_tokens.csv",
    "png_ground_truth_summary": PAIRING_ROOT / "E6_alkafri_png_ground_truth_summary.csv",
    "pairing_candidates": PAIRING_ROOT / "E6_alkafri_pairing_candidates.csv",
    "pairing_report": PAIRING_ROOT / "E6_alkafri_pairing_report.json",
    "pairing_validation_report": PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_validation_report.json",
    "pairing_validation_sanity": PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_validation_sanity_checks.json",
    "source_metadata_files": PAIRING_VALIDATION_ROOT / "E6_alkafri_source_metadata_files.csv",
    "source_metadata_preview": PAIRING_VALIDATION_ROOT / "E6_alkafri_source_metadata_tables_preview.csv",
    "source_code_snippets": PAIRING_VALIDATION_ROOT / "E6_alkafri_source_code_relevant_snippets.csv",
    "image_specific_tokens": PAIRING_VALIDATION_ROOT / "E6_alkafri_image_specific_tokens.csv",
    "gt_specific_tokens": PAIRING_VALIDATION_ROOT / "E6_alkafri_gt_specific_tokens.csv",
    "final_gt_tokens": PAIRING_VALIDATION_ROOT / "E6_alkafri_final_gt_png_tokens.csv",
    "axial_series_by_case": PAIRING_VALIDATION_ROOT / "E6_alkafri_axial_series_by_case.csv",
}

def read_csv_optional(path):
    return pd.read_csv(path) if Path(path).exists() else pd.DataFrame()

extracted_inventory_df = read_csv_optional(INPUTS["extracted_inventory"])
axial_ima_df = read_csv_optional(INPUTS["axial_ima_candidates"])
axial_series_summary_df = read_csv_optional(INPUTS["axial_series_summary"])
gt_real_df = read_csv_optional(INPUTS["ground_truth_real_files"])
image_specific_tokens_df = read_csv_optional(INPUTS["image_specific_tokens"])
gt_specific_tokens_df = read_csv_optional(INPUTS["gt_specific_tokens"])
final_gt_tokens_df = read_csv_optional(INPUTS["final_gt_tokens"])
source_metadata_preview_df = read_csv_optional(INPUTS["source_metadata_preview"])
pairing_candidates_df = read_csv_optional(INPUTS["pairing_candidates"])

def read_json_optional(path):
    if Path(path).exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

inventory_report = read_json_optional(INPUTS["inventory_report"])
pairing_report = read_json_optional(INPUTS["pairing_report"])
pairing_validation_report = read_json_optional(INPUTS["pairing_validation_report"])
pairing_validation_sanity = read_json_optional(INPUTS["pairing_validation_sanity"])

previous_summary = {
    "extracted_inventory_rows": int(len(extracted_inventory_df)),
    "axial_ima_candidates": int(len(axial_ima_df)),
    "axial_series_summary": int(len(axial_series_summary_df)),
    "ground_truth_real_files": int(len(gt_real_df)),
    "image_specific_tokens": int(len(image_specific_tokens_df)),
    "gt_specific_tokens": int(len(gt_specific_tokens_df)),
    "final_gt_tokens": int(len(final_gt_tokens_df)),
    "pairing_candidates": int(len(pairing_candidates_df)),
    "pairing_validation_decision": pairing_validation_report.get("pairing_v1_decision"),
    "pairing_validation_recommendation": pairing_validation_report.get("recommendation_for_notebook_14"),
}
previous_outputs_summary_json_path = CURATION_ROOT / "E7_alkafri_previous_outputs_summary.json"
with open(previous_outputs_summary_json_path, "w", encoding="utf-8") as f:
    json.dump(previous_summary, f, indent=2, ensure_ascii=False)
print(json.dumps(previous_summary, indent=2, ensure_ascii=False))

## Diagnostico de estado previo

In [ ]:
prior_status_rows = [
    {"metric": "valid_ima_inventory", "value": inventory_report.get("valid_dicom_files", None)},
    {"metric": "axial_candidates", "value": len(axial_ima_df)},
    {"metric": "axial_series", "value": len(axial_series_summary_df)},
    {"metric": "ground_truth_png", "value": int(gt_real_df.get("extension", pd.Series(dtype=str)).astype(str).str.lower().eq(".png").sum()) if len(gt_real_df) else 0},
    {"metric": "final_ground_truth", "value": len(final_gt_tokens_df)},
    {"metric": "notebook_13b_decision", "value": pairing_validation_report.get("pairing_v1_decision", "unknown")},
    {"metric": "why_pairing_v1_not_reliable", "value": "shape/tokens generated many ambiguous candidates; manual/structural validation required"},
    {"metric": "why_pairing_refined_v1_not_created", "value": "; ".join(pairing_validation_sanity.get("problems", [])) if pairing_validation_sanity else "unknown"},
]
prior_status_df = pd.DataFrame(prior_status_rows)
prior_status_csv_path = CURATION_ROOT / "E7_alkafri_axial_prior_status.csv"
prior_status_df.to_csv(prior_status_csv_path, index=False)
display(prior_status_df)

## Reconstruccion de estructura de casos

In [ ]:
def extract_case_id(text):
    text = str(text)
    m = re.search(r"01_MRI_Data[/\\]([0-9]{4})_", text)
    if m:
        return m.group(1)
    m = re.search(r"[_/\\]([0-9]{4})[_/\\]?", text)
    return m.group(1) if m else None

def extract_modality(text):
    t = str(text).lower()
    if re.search(r"\bt1\b|t1_", t):
        return "T1"
    if re.search(r"\bt2\b|t2_", t):
        return "T2"
    return None

def extract_disc_or_slice(text):
    m = re.search(r"\bD[1-9]\b", str(text), flags=re.I)
    if m:
        return m.group(0).upper()
    nums = re.findall(r"\d+", str(text))
    return nums[-1] if nums else None

def extract_gt_type(text):
    t = str(text).lower()
    if "05_final_ground_truth_data" in t or "final" in t:
        return "final"
    if "04_intermediary_ground_truth_data" in t or "intermediary" in t:
        return "intermediary"
    if "03_manual_label_data" in t or "manual" in t:
        return "manual"
    return "unknown"

def extract_labeller(text):
    m = re.search(r"labeller\s*([0-9]+)", str(text), flags=re.I)
    return m.group(1) if m else None

image_case_rows = []
for _, row in axial_ima_df.iterrows():
    text = f"{row.get('relative_path', '')} {row.get('file_path', '')} {row.get('SeriesDescription', '')}"
    image_case_rows.append({
        "image_file_path": row.get("file_path"),
        "image_relative_path": row.get("relative_path"),
        "case_id": row.get("case_id") if "case_id" in row and pd.notna(row.get("case_id")) else extract_case_id(text),
        "modality": row.get("modality") if "modality" in row and pd.notna(row.get("modality")) else extract_modality(text),
        "disc_or_slice_id": extract_disc_or_slice(text),
        "series_id": row.get("SeriesInstanceUID"),
        "instance_number": row.get("InstanceNumber"),
        "series_description": row.get("SeriesDescription"),
        "is_posdisp_or_localizer": bool(re.search(r"posdisp|localizer|localiser", text, flags=re.I)),
    })
image_case_index_df = pd.DataFrame(image_case_rows)
image_case_index_csv_path = CURATION_ROOT / "E7_alkafri_axial_image_case_index.csv"
image_case_index_df.to_csv(image_case_index_csv_path, index=False)

gt_case_rows = []
source_gt_df = gt_specific_tokens_df if len(gt_specific_tokens_df) else gt_real_df
for _, row in source_gt_df.iterrows():
    text = f"{row.get('relative_path', '')} {row.get('file_path', '')} {row.get('file_name', '')}"
    gt_case_rows.append({
        "gt_file_path": row.get("file_path"),
        "gt_relative_path": row.get("relative_path"),
        "case_id": row.get("case_id") if "case_id" in row and pd.notna(row.get("case_id")) else extract_case_id(text),
        "modality": row.get("modality") if "modality" in row and pd.notna(row.get("modality")) else extract_modality(text),
        "disc_or_slice_id": row.get("disc_id") if "disc_id" in row and pd.notna(row.get("disc_id")) else extract_disc_or_slice(text),
        "labeller": row.get("labeller") if "labeller" in row else extract_labeller(text),
        "gt_type": row.get("gt_type") if "gt_type" in row and pd.notna(row.get("gt_type")) else extract_gt_type(text),
        "extension": row.get("extension"),
    })
gt_case_index_df = pd.DataFrame(gt_case_rows)
gt_case_index_csv_path = CURATION_ROOT / "E7_alkafri_gt_case_index.csv"
gt_case_index_df.to_csv(gt_case_index_csv_path, index=False)
display(image_case_index_df.head())
display(gt_case_index_df.head())

## Seleccion conservadora de candidatos

In [ ]:
def read_ima_shape(path):
    try:
        ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        return (int(ds.Rows), int(ds.Columns))
    except Exception:
        return None

def read_mask_shape(path):
    try:
        with Image.open(path) as img:
            w, h = img.size
        return (h, w)
    except Exception:
        return None

image_filtered_df = image_case_index_df[
    image_case_index_df["case_id"].notna()
    & image_case_index_df["modality"].notna()
    & ~image_case_index_df["is_posdisp_or_localizer"].fillna(False)
].copy()
gt_filtered_df = gt_case_index_df[
    gt_case_index_df["case_id"].notna()
    & gt_case_index_df["modality"].notna()
    & gt_case_index_df["extension"].astype(str).str.lower().eq(".png")
    & gt_case_index_df["gt_type"].isin(["final", "intermediary"])
].copy()

strict_rows = []
for _, img in tqdm(image_filtered_df.iterrows(), total=len(image_filtered_df), desc="Strict candidates"):
    matches = gt_filtered_df[
        gt_filtered_df["case_id"].astype(str).eq(str(img["case_id"]))
        & gt_filtered_df["modality"].astype(str).eq(str(img["modality"]))
    ].copy()
    if len(matches) == 0:
        continue
    img_shape = read_ima_shape(img["image_file_path"])
    for _, gt in matches.iterrows():
        mask_shape = read_mask_shape(gt["gt_file_path"])
        shape_match = bool(img_shape is not None and mask_shape is not None and tuple(img_shape) == tuple(mask_shape))
        slice_match = bool(pd.notna(gt.get("disc_or_slice_id")) and str(gt.get("disc_or_slice_id")) != "")
        evidence_source = bool(len(source_metadata_preview_df) > 0)
        signals = [True, True, slice_match, shape_match, evidence_source]
        if gt["gt_type"] == "final" and shape_match and slice_match:
            confidence = "alta" if evidence_source else "media"
        elif shape_match and gt["gt_type"] in ["final", "intermediary"]:
            confidence = "media"
        else:
            confidence = "baja"
        strict_rows.append({
            "image_file_path": img["image_file_path"],
            "gt_file_path": gt["gt_file_path"],
            "image_relative_path": img["image_relative_path"],
            "gt_relative_path": gt["gt_relative_path"],
            "case_id": img["case_id"],
            "modality": img["modality"],
            "gt_type": gt["gt_type"],
            "disc_or_slice_id": gt["disc_or_slice_id"],
            "instance_number": img["instance_number"],
            "image_shape": str(img_shape),
            "mask_shape": str(mask_shape),
            "evidence_case_match": True,
            "evidence_modality_match": True,
            "evidence_slice_match": slice_match,
            "evidence_shape_match": shape_match,
            "evidence_source_metadata": evidence_source,
            "confidence_previsual": confidence,
            "reason": json.dumps({"gt_type": gt["gt_type"], "shape_match": shape_match, "slice_match": slice_match}),
            "needs_manual_review": confidence != "alta",
        })
strict_candidate_pairs_df = pd.DataFrame(strict_rows)
strict_candidate_pairs_csv_path = CURATION_ROOT / "E7_alkafri_axial_strict_candidate_pairs.csv"
strict_candidate_pairs_df.to_csv(strict_candidate_pairs_csv_path, index=False)
print("Strict candidates:", len(strict_candidate_pairs_df))
display(strict_candidate_pairs_df.head())

## Visualizacion para curacion y sanity automatico

In [ ]:
def normalize_image(arr):
    arr = arr.astype(np.float32)
    lo, hi = np.percentile(arr, [1, 99])
    if np.isclose(lo, hi):
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - lo) / (hi - lo), 0, 1).astype(np.float32)

def read_image_ima(path):
    ds = pydicom.dcmread(str(path), force=True)
    return normalize_image(ds.pixel_array), ds

def read_mask_png(path):
    arr = np.asarray(Image.open(path))
    if arr.ndim == 3:
        return np.any(arr[:, :, :3] != 0, axis=2).astype(np.uint8)
    return arr

def components_count(mask_bool):
    try:
        return int(ndimage.label(mask_bool)[1])
    except Exception:
        return None

sanity_rows = []
review_rows = []
candidate_subset = strict_candidate_pairs_df[strict_candidate_pairs_df["confidence_previsual"].isin(["alta", "media"])].head(MAX_CURATION_CANDIDATES).copy() if len(strict_candidate_pairs_df) else pd.DataFrame()

for idx, row in enumerate(candidate_subset.itertuples(index=False), start=1):
    candidate_id = f"cur_{idx:03d}"
    fig_path = FIGURES_ROOT / f"E7_alkafri_curated_candidate_{idx:03d}.png"
    overlay_generated = False
    error = None
    try:
        image, ds = read_image_ima(row.image_file_path)
        mask = read_mask_png(row.gt_file_path)
        resize_needed = tuple(image.shape[:2]) != tuple(mask.shape[:2])
        mask_vis = resize(mask, image.shape[:2], order=0, preserve_range=True, anti_aliasing=False).astype(mask.dtype) if resize_needed else mask
        fg = mask_vis != 0
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(image, cmap="gray")
        axes[1].imshow(mask, cmap="viridis")
        axes[2].imshow(image, cmap="gray")
        axes[2].imshow(np.ma.masked_where(mask_vis == 0, mask_vis), cmap="autumn", alpha=0.45)
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"{candidate_id} {row.case_id} {row.modality} {row.gt_type} {row.disc_or_slice_id} Inst {row.instance_number}")
        fig.tight_layout()
        fig.savefig(fig_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        overlay_generated = True
        fg_ratio = float(np.mean(fg))
        mask_empty = bool(fg_ratio == 0)
        mask_full = bool(fg_ratio > 0.95)
        image_valid = bool(np.isfinite(image).all() and np.std(image) > 0)
        shape_match = not resize_needed
        n_components = components_count(fg)
    except Exception as exc:
        fg_ratio = None
        mask_empty = None
        mask_full = None
        image_valid = False
        shape_match = False
        resize_needed = None
        n_components = None
        error = repr(exc)

    auto_sanity_status = "ok" if overlay_generated and image_valid and (fg_ratio is not None and 0 < fg_ratio < 0.95) else "review"
    sanity_rows.append({
        "candidate_id": candidate_id,
        "image_file_path": row.image_file_path,
        "gt_file_path": row.gt_file_path,
        "mask_foreground_ratio": fg_ratio,
        "mask_empty": mask_empty,
        "mask_full": mask_full,
        "image_valid_range": image_valid,
        "shape_match": shape_match,
        "connected_components": n_components,
        "overlay_generated": overlay_generated,
        "resize_needed": resize_needed,
        "error": error,
        "auto_sanity_status": auto_sanity_status,
    })
    review_rows.append({
        "candidate_id": candidate_id,
        "figure_path": str(fig_path),
        "image_file_path": row.image_file_path,
        "gt_file_path": row.gt_file_path,
        "case_id": row.case_id,
        "modality": row.modality,
        "gt_type": row.gt_type,
        "disc_or_slice_id": row.disc_or_slice_id,
        "instance_number": row.instance_number,
        "confidence_previsual": row.confidence_previsual,
        "auto_sanity_status": auto_sanity_status,
        "manual_accept": "",
        "manual_reject_reason": "",
        "notes": "",
    })

candidate_sanity_df = pd.DataFrame(sanity_rows)
curation_review_sheet_df = pd.DataFrame(review_rows)
candidate_sanity_csv_path = CURATION_ROOT / "E7_alkafri_axial_candidate_sanity_checks.csv"
curation_review_sheet_csv_path = CURATION_ROOT / "E7_alkafri_axial_curation_review_sheet.csv"
candidate_sanity_df.to_csv(candidate_sanity_csv_path, index=False)
curation_review_sheet_df.to_csv(curation_review_sheet_csv_path, index=False)
display(candidate_sanity_df.head())

## Curacion semiautomatica y revision manual opcional

In [ ]:
auto_df = curation_review_sheet_df.merge(candidate_sanity_df, on=["candidate_id", "image_file_path", "gt_file_path"], how="left")
auto_df["auto_accept_candidate"] = (
    auto_df["confidence_previsual"].isin(["alta", "media"])
    & auto_df["gt_type"].isin(["final", "intermediary"])
    & auto_df["overlay_generated"].fillna(False)
    & auto_df["image_valid_range"].fillna(False)
    & ~auto_df["mask_empty"].fillna(True)
    & ~auto_df["mask_full"].fillna(True)
    & auto_df["auto_sanity_status"].eq("ok")
)
auto_curated_csv_path = CURATION_ROOT / "E7_alkafri_axial_auto_curated_candidates.csv"
auto_df.to_csv(auto_curated_csv_path, index=False)

manual_review_path = CURATION_ROOT / "E7_alkafri_axial_curation_review_sheet_MANUAL.csv"
if manual_review_path.exists():
    manual_df = pd.read_csv(manual_review_path)
    accepted_df = auto_df.merge(manual_df[["candidate_id", "manual_accept", "manual_reject_reason", "notes"]], on="candidate_id", how="left", suffixes=("", "_manual"))
    accepted_df = accepted_df[accepted_df["manual_accept_manual"].astype(str).str.lower().eq("yes")].copy()
    curation_source = "manual_review"
else:
    accepted_df = auto_df[auto_df["auto_accept_candidate"]].copy()
    curation_source = "curated_automatic_preliminary"

print("Auto accepted:", int(auto_df["auto_accept_candidate"].sum()))
print("Accepted for dataset:", len(accepted_df), "source:", curation_source)

## Crear subset axial curado si corresponde

In [ ]:
processed_rows = []
curated_dataset_created = False
if len(accepted_df) >= MIN_CURATED_PAIRS:
    CURATED_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
    for idx, row in enumerate(accepted_df.itertuples(index=False), start=1):
        sample_id = f"alkafri_axial_curated_{idx:04d}"
        try:
            image, ds = read_image_ima(row.image_file_path)
            mask = read_mask_png(row.gt_file_path)
            image_out = CURATED_PROCESSED_ROOT / f"{sample_id}_image.npy"
            mask_out = CURATED_PROCESSED_ROOT / f"{sample_id}_mask.npy"
            meta_out = CURATED_PROCESSED_ROOT / f"{sample_id}_metadata.json"
            np.save(image_out, image.astype(np.float32))
            np.save(mask_out, mask)
            metadata = {
                "sample_id": sample_id,
                "image_file_path": row.image_file_path,
                "gt_file_path": row.gt_file_path,
                "case_id": row.case_id,
                "modality": row.modality,
                "gt_type": row.gt_type,
                "disc_or_slice_id": row.disc_or_slice_id,
                "instance_number": str(row.instance_number),
                "curation_source": curation_source,
                "confidence": row.confidence_previsual,
                "notes": getattr(row, "notes", ""),
            }
            with open(meta_out, "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=2, ensure_ascii=False)
            processed_rows.append({
                **metadata,
                "image_npy": str(image_out),
                "mask_npy": str(mask_out),
                "metadata_json": str(meta_out),
                "image_shape": str(tuple(image.shape)),
                "mask_shape": str(tuple(mask.shape)),
            })
        except Exception as exc:
            processed_rows.append({"sample_id": sample_id, "image_file_path": row.image_file_path, "gt_file_path": row.gt_file_path, "error": repr(exc)})
    curated_dataset_created = any("image_npy" in row for row in processed_rows)
else:
    print("Menos de MIN_CURATED_PAIRS; no se crea dataset entrenable.")

curated_index_df = pd.DataFrame(processed_rows)
curated_index_csv_path = CURATION_ROOT / "E7_alkafri_axial_curated_v1_index.csv"
curated_index_df.to_csv(curated_index_csv_path, index=False)
display(curated_index_df.head())

## Visualizacion final y resumen del subset

In [ ]:
curated_example_paths = []
if len(curated_index_df) > 0 and "image_npy" in curated_index_df.columns:
    for idx, row in enumerate(curated_index_df.dropna(subset=["image_npy", "mask_npy"]).head(30).itertuples(index=False), start=1):
        try:
            image = np.load(row.image_npy)
            mask = np.load(row.mask_npy)
            mask_vis = resize(mask, image.shape[:2], order=0, preserve_range=True, anti_aliasing=False).astype(mask.dtype) if tuple(mask.shape[:2]) != tuple(image.shape[:2]) else mask
            fig_path = FIGURES_ROOT / f"E7_alkafri_axial_curated_v1_example_{idx:02d}.png"
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(image, cmap="gray")
            axes[1].imshow(mask, cmap="viridis")
            axes[2].imshow(image, cmap="gray")
            axes[2].imshow(np.ma.masked_where(mask_vis == 0, mask_vis), cmap="autumn", alpha=0.45)
            for ax in axes:
                ax.axis("off")
            fig.suptitle(f"{row.sample_id} {row.case_id} {row.modality} {row.gt_type}")
            fig.tight_layout()
            fig.savefig(fig_path, dpi=160, bbox_inches="tight")
            plt.close(fig)
            curated_example_paths.append(str(fig_path))
        except Exception as exc:
            print("Error figura curated:", repr(exc))

summary_rows = []
if len(accepted_df) > 0:
    summary_rows.append({
        "total_accepted_pairs": int(len(accepted_df)),
        "unique_cases": int(accepted_df["case_id"].nunique()),
        "modalities": json.dumps(accepted_df["modality"].value_counts().to_dict(), ensure_ascii=False),
        "gt_type": json.dumps(accepted_df["gt_type"].value_counts().to_dict(), ensure_ascii=False),
        "foreground_ratio_mean": float(candidate_sanity_df["mask_foreground_ratio"].dropna().mean()) if "mask_foreground_ratio" in candidate_sanity_df else None,
        "empty_masks": int(candidate_sanity_df["mask_empty"].fillna(False).sum()) if "mask_empty" in candidate_sanity_df else 0,
        "full_masks": int(candidate_sanity_df["mask_full"].fillna(False).sum()) if "mask_full" in candidate_sanity_df else 0,
        "disc_or_slice_counts": json.dumps(accepted_df["disc_or_slice_id"].value_counts(dropna=False).to_dict(), ensure_ascii=False),
        "curated_dataset_created": bool(curated_dataset_created),
    })
curated_summary_df = pd.DataFrame(summary_rows)
curated_summary_csv_path = CURATION_ROOT / "E7_alkafri_axial_curated_v1_summary.csv"
curated_summary_df.to_csv(curated_summary_csv_path, index=False)
display(curated_summary_df)

## Decision tecnica y reporte

In [ ]:
license_ok = bool(dataset_license_audit_df[dataset_license_audit_df["dataset_name"].str.contains("Lumbar Spine MRI Dataset", regex=False)]["license"].eq("CC BY 4.0").any())
feasibility = {
    "license_ok": license_ok,
    "redistributable_under_cc_by": True,
    "has_native_axial": True,
    "has_ground_truth": bool(len(gt_case_index_df) > 0),
    "previous_pairing_was_ambiguous": True,
    "strict_candidates_count": int(len(strict_candidate_pairs_df)),
    "auto_curated_count": int(auto_df["auto_accept_candidate"].sum()) if len(auto_df) else 0,
    "manual_curated_count": int(len(accepted_df)) if curation_source == "manual_review" else 0,
    "curated_dataset_created": bool(curated_dataset_created),
    "curated_dataset_path": str(CURATED_PROCESSED_ROOT) if curated_dataset_created else None,
    "usable_for_axial_segmentation_baseline": bool(curated_dataset_created and len(accepted_df) >= MIN_CURATED_PAIRS),
    "recommended_next_step": "notebook_16_axial_baseline_segmentation" if curated_dataset_created and len(accepted_df) >= MIN_CURATED_PAIRS else "manual_review_or_continue_dataset_search",
}
feasibility_json_path = CURATION_ROOT / "E7_alkafri_axial_curated_feasibility_assessment.json"
with open(feasibility_json_path, "w", encoding="utf-8") as f:
    json.dump(feasibility, f, indent=2, ensure_ascii=False)

report = {
    "license_audit": dataset_license_audit_df.to_dict(orient="records"),
    "datasets_source": ["Al-Kafri/Sudirman MRI", "Al-Kafri/Sudirman Label Image Ground Truth"],
    "outputs_used": {k: str(v) for k, v in INPUTS.items()},
    "strict_candidates": int(len(strict_candidate_pairs_df)),
    "figures": curation_review_sheet_df.get("figure_path", pd.Series(dtype=str)).dropna().tolist() + curated_example_paths,
    "sanity_checks": str(candidate_sanity_csv_path),
    "subset_created": bool(curated_dataset_created),
    "technical_decision": feasibility,
    "limitations": [
        "Curacion automatica preliminar no reemplaza revision manual/profesional.",
        "No se promete validez clinica.",
        "No se entrenan modelos en este notebook.",
        "No se usan pares de baja confianza.",
    ],
}
report_json_path = CURATION_ROOT / "E7_alkafri_axial_license_and_curated_subset_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(feasibility, indent=2, ensure_ascii=False))

## Conclusion Markdown

In [ ]:
license_table_md = dataset_license_audit_df.to_markdown(index=False)
summary_md = curated_summary_df.to_markdown(index=False) if len(curated_summary_df) else "No se creo subset curado entrenable."

conclusion_text = f'''# Conclusion tecnica - E7 Al-Kafri axial licencia y subset curado

## Por que se hizo este notebook

El plano axial es clinicamente relevante segun user research, pero los notebooks 13/13b mostraron que el pairing automatico masivo era ambiguo. Este notebook audita licencia y construye un subset axial curado solo si hay evidencia estructural y visual suficiente.

## Alineacion con fuentes de datos UADE

Se priorizan datasets publicos, descargables sin permisos especiales y con licencia permisiva. No se usa RSNA/LumbarDISC como dataset experimental principal por restricciones MIRA/Kaggle/no comerciales.

## Auditoria de licencia

{license_table_md}

Los datasets Al-Kafri/Sudirman de MRI, labels y notas estan documentados como CC BY 4.0. El codigo MATLAB esta bajo GPLv3 y se usa solo como referencia de estructura, no como codigo de producto.

## Resultado de curacion

{summary_md}

Subset curado creado: {curated_dataset_created}.

## Decision sobre entrenamiento axial

Usable para baseline axial: {feasibility["usable_for_axial_segmentation_baseline"]}.
Siguiente paso recomendado: `{feasibility["recommended_next_step"]}`.

## Limitaciones

- No se entrenan modelos.
- No se promete validez clinica.
- La curacion automatica preliminar requiere revision manual/profesional.
- No se llaman ground truth a pares ambiguos.
- Si no se alcanza el umbral minimo, corresponde revision manual o busqueda adicional de dataset.
'''

conclusion_path = DOCS_ROOT / "E7_alkafri_axial_license_and_curated_subset_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)
print("Conclusion:", conclusion_path)

## Criterio de aceptacion

Este notebook audita licencia, usa fuentes publicas/permisivas, no entrena modelos, no usa RSNA/Kaggle/MIRA como dataset experimental, reconstruye indices de caso, genera candidatos estrictos, exporta figuras de curacion, crea una review sheet editable, crea subset curado solo con suficientes pares confiables y exporta CSV/JSON/Markdown.